# AlphaTrain Pillar 2Y: V11 policy-only training

Second iteration of policy-only training on a V11 self-play corpus. Warm-starts from pillar2x2 epoch 10 (V10 winner, +19% over 2W2; lr=3e-4 was the better of the two parallel runs).

**Data:** V11 with feature-value MCTS, **7.78M states** from:
- 694 anchor self-play games @ 600 sims, cap 6500 turns (3.05M states, mean 9,101 / median 10,111 / 40% capped)
- 7,811 prevention crisis replays @ 600 sims, rewind 30, continue 500 (2.83M states, mean 3,502 / 67% capped)
- 7,810 recovery crisis replays @ 600 sims, rewind 15, continue 500 (1.90M states, mean 3,273 / 44% capped)

Built with `build_expert_v2_tensor.py --policy-only-data` (V10+ writes bootstrap_value=0 for capped games; the builder requires this flag to acknowledge those targets are unused at training time).

**Model:** Warm-start from pillar2x2 epoch 10. Backbone + policy head transfer; `--policy-only` builds an architecture without the value head so the loader filters value_* keys automatically.

**Hyperparameters:** lr=3e-4 (V10's winning rate), 20 epochs, 2 warmup epochs, batch=32768. (2X2 was still improving at epoch 10 — give 2Y room to keep learning, save every epoch so you can stop / eval intermediate.)

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code (~64 KB)
2. `selfplay_v11.pt.gz` — V11 tensor (~600 MB compressed, 4.3 GB unpacked)
3. `pillar2x2_epoch_10.pt` — warm-start weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar2x2_epoch_10.pt', '/content/alphatrain/data/model.pt')
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v11.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2Y: V11 data (7.78M states), warm-start from 2X2 ep10, policy-only.
# lr=3e-4 won the V10 2X vs 2X2 A/B; reuse it here.
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --policy-only \
    --epochs 20 --batch-size 32768 --lr 3e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2y_best.pt \
    --save-dir /content/checkpoints/pillar2y

In [ ]:
# Copy ALL epoch checkpoints to Drive
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar2y/epoch_*.pt')):
    dst = f'{DRIVE}/pillar2y_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2y/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/pillar2y_{f}')